In [0]:

import pyspark.sql.functions as F
import pandas as pd
from datetime import date, timedelta

yesterday = (date.today() - timedelta(days=1)).strftime("%Y%m%d")

T_trans    = spark.table("cpx_dataanalytics_gold.customer360.c360_f_transaction")
T_location = spark.table("cpx_dataanalytics_gold.customer360.cpx_d_location")
T_item     = spark.table("cpx_dataanalytics_gold.customer360.cpx_d_item")

# Category sets — identical to the FIFA notebook (PDM gaming lands under TRR - AMUSEMENT codes)
games_categories = {
    # "TRR - AMUSEMENT - G&E",
    "TRR - AMUSEMENT - GAMES",
    "TRR - AMUSEMENT - PHYSICAL GAMES",
    "TRR - PROMO - AMUSEMENT",
}
food_categories = {
    # "MISC", 
    "PDM - MENU FOOD - CANDY", 
    # "PDM - MENU FOOD - G&E",
    "TRR - INV ALC BEV - BOTTLE", 
    "TRR - INV ALC BEV - CAN", 
    "TRR - INV ALC BEV - WINE",
    "TRR - MENU ALC BEV - COCKTAIL", 
    "TRR - MENU ALC BEV - DRAUGHT BEER",
    "TRR - MENU BEV - NA COLD BEV", 
    "TRR - MENU BEV - NA HOT BEV",
    "TRR - MENU FOOD - APP", 
    "TRR - MENU FOOD - DESSERT", 
    # "TRR - MENU FOOD - G&E",
    "TRR - MENU FOOD - KIDS", 
    "TRR - MENU FOOD - PIZZA", 
    "TRR - MENU FOOD - POUTINE",
    "TRR - MENU FOOD - SANDWICHES", 
    "TRR - MOD MENU - ADD-ON", 
    # "TRR - MOD MENU - NO VALUE",
    # "CHARITY",
    # "COUPONS", 
    "TRR - INV BEV - NA BEV", 
    "TRR - MENU ALC BEV - LIQUOR",
    "TRR - MENU ALC BEV - WINE", 
    "TRR - MENU FOOD - ENTRÉE", 
    "TRR - MENU FOOD - SALAD",
    "VHO-LICENSED FOOD", 
    # "Box Office Surcharges and Fees", 
    "TRR - INV ALC BEV - LIQUOR",
    "VHO-CANDY", 
    "VIP Food - Addons", 
    "VHO-LICENSED DRINKS", 
    "TRR - MENU FOOD - BRUNCH",
    # "TRR - MENU FOOD - G&E OLD",
    #  "VHO-BIRTHDAY", 
     "VHO-DRINK",
}


In [0]:
# ── Source DataFrame: item-line grain, carries CDE_ID + LocationId for visit tuple ──
pdm_tx = (
    T_trans
    .join(T_location, T_trans.LocationId == T_location.LocationID, "inner")
    .join(T_item, T_trans.ItemSkey == T_item.itemskey, "inner")
    .filter(F.col("LC_Location_Type_Description").like("Restaurant%"))
    .filter(F.col("LC_Trade_Brand_Description").isin("Rec Room", "Playdium"))
    .filter((F.col("VisitDateId") >= 20250101) & (F.col("VisitDateId") <= int(yesterday)))
    .filter(F.col("Item_Class_Description1").isin(list(games_categories) + list(food_categories)))
    .withColumn(
        "category_label",
        F.when(F.col("Item_Class_Description1").isin(list(games_categories)), F.lit("Games"))
         .when(F.col("Item_Class_Description1").isin(list(food_categories)),  F.lit("Food"))
    )
    .filter(F.col("category_label").isNotNull())
    .select(
        "Location_Name",
        T_trans.LocationId.alias("LocationId"),   # qualified: both tables have LocationId
        F.col("LC_Trade_Brand_Description").alias("brand"),
        "CDE_ID", "VisitDateId",
        "category_label", "Net_Revenue","Transaction_Number"
    )
)


In [0]:

# ── VERIFY: per venue — item rows, identified visits, customers, date span ──
print("=== Per-venue counts + date span (identified customers only for visits) ===")
(pdm_tx
   .filter(F.col("CDE_ID").isNotNull())
   .groupBy("brand", "Location_Name")
   .agg(F.count("*").alias("item_rows"),
        F.countDistinct(F.concat_ws("|", F.col("CDE_ID").cast("string"), F.col("VisitDateId").cast("string"))).alias("identified_visits"),
        F.countDistinct("CDE_ID").alias("customers"),
        F.min("VisitDateId").alias("first_date"),
        F.max("VisitDateId").alias("last_date"))
   .orderBy("brand", "Location_Name")
   .show(truncate=False))

# ── CDE_ID coverage — visits only count identified customers; this is the ceiling ──
total   = pdm_tx.count()
nonnull = pdm_tx.filter(F.col("CDE_ID").isNotNull()).count()
print(f"=== CDE_ID coverage: {nonnull:,}/{total:,} item rows = {100*nonnull/total:.1f}% identified "
      f"({100*(total-nonnull)/total:.1f}% guest/anonymous) ===")
print("Note: 'identified_visits' excludes guests → treat as a LOWER BOUND on true footfall.")

In [0]:
# ═════════════════════════════════════════════════════════════════════════════
# CELL 4 — Quarterly F&B vs Gaming Trend by Brand (Rec Room & Playdium)
# Revenue + transaction counts by quarter, category, and brand.
# Q3 2026 is partial (through Aug 6) — flagged in output.
# ═════════════════════════════════════════════════════════════════════════════

# Derive quarter from VisitDateId (integer YYYYMMDD)
quarterly = (
    pdm_tx
    .filter(F.col("VisitDateId") >= 20250101)
    .withColumn("year", (F.col("VisitDateId") / 10000).cast("int"))
    .withColumn("month", ((F.col("VisitDateId") / 100) % 100).cast("int"))
    .withColumn("quarter", F.concat(F.lit("Q"), F.ceil(F.col("month") / 3).cast("int")))
    .groupBy("year", "quarter", "brand", "category_label")
    .agg(
        F.sum(F.col("Net_Revenue").cast("double")).alias("total_revenue"),
        F.countDistinct("Transaction_Number").alias("distinct_transactions"),
        F.count("*").alias("item_rows"),
    )
    .orderBy("year", "quarter", "brand", "category_label")
)

df_q = quarterly.toPandas()

# Display per brand
for brand_name in ["Rec Room", "Playdium"]:
    df_brand = df_q[df_q["brand"] == brand_name].copy()
    df_brand["total_revenue"] = df_brand["total_revenue"].map(lambda v: f"${v:,.0f}")
    df_brand["distinct_transactions"] = df_brand["distinct_transactions"].map(lambda v: f"{v:,}")
    df_brand["item_rows"] = df_brand["item_rows"].map(lambda v: f"{v:,}")
    print("═" * 90)
    print(f"QUARTERLY F&B vs GAMING TREND — {brand_name.upper()} (2025 & 2026)")
    print("═" * 90)
    print(df_brand.rename(columns={
        "year": "Year", "quarter": "Quarter", "brand": "Brand", "category_label": "Category",
        "total_revenue": "Revenue", "distinct_transactions": "Transactions",
        "item_rows": "Item Rows"
    }).drop(columns=["Brand"]).to_string(index=False))
    print()
print("⚠️  Q3 is partial (Jul 1 – Aug 6 only).")

In [0]:
# ═════════════════════════════════════════════════════════════════════════════
# CELL 5 — Scene members only (SceneFlag='Yes'), by Brand
# ═════════════════════════════════════════════════════════════════════════════

pdm_tx_scene = (
    T_trans
    .filter(F.col("SCENEFLAG") == "Yes")
    .join(T_location, T_trans.LocationId == T_location.LocationID, "inner")
    .join(T_item, T_trans.ItemSkey == T_item.itemskey, "inner")
    .filter(F.col("LC_Location_Type_Description").like("Restaurant%"))
    .filter(F.col("LC_Trade_Brand_Description").isin("Rec Room", "Playdium"))
    .filter((F.col("VisitDateId") >= 20250101) & (F.col("VisitDateId") <= int(yesterday)))
    .filter(F.col("Item_Class_Description1").isin(list(games_categories) + list(food_categories)))
    .withColumn(
        "category_label",
        F.when(F.col("Item_Class_Description1").isin(list(games_categories)), F.lit("Games"))
         .when(F.col("Item_Class_Description1").isin(list(food_categories)),  F.lit("Food"))
    )
    .filter(F.col("category_label").isNotNull())
    .select("Location_Name", T_trans.LocationId.alias("LocationId"),
            F.col("LC_Trade_Brand_Description").alias("brand"),
            "CDE_ID", "VisitDateId", "category_label", "Net_Revenue", "Transaction_Number")
)

quarterly_scene = (
    pdm_tx_scene
    .withColumn("year", (F.col("VisitDateId") / 10000).cast("int"))
    .withColumn("month", ((F.col("VisitDateId") / 100) % 100).cast("int"))
    .withColumn("quarter", F.concat(F.lit("Q"), F.ceil(F.col("month") / 3).cast("int")))
    .groupBy("year", "quarter", "brand", "category_label")
    .agg(
        F.sum(F.col("Net_Revenue").cast("double")).alias("total_revenue"),
        F.countDistinct("Transaction_Number").alias("distinct_transactions"),
        F.count("*").alias("item_rows"),
    )
    .orderBy("year", "quarter", "brand", "category_label")
)

df_qs = quarterly_scene.toPandas()

for brand_name in ["Rec Room", "Playdium"]:
    df_brand = df_qs[df_qs["brand"] == brand_name].copy()
    df_brand["total_revenue"] = df_brand["total_revenue"].map(lambda v: f"${v:,.0f}")
    df_brand["distinct_transactions"] = df_brand["distinct_transactions"].map(lambda v: f"{v:,}")
    df_brand["item_rows"] = df_brand["item_rows"].map(lambda v: f"{v:,}")
    print("═" * 90)
    print(f"QUARTERLY F&B vs GAMING — SCENE MEMBERS — {brand_name.upper()} (2025 & 2026)")
    print("═" * 90)
    print(df_brand.rename(columns={
        "year": "Year", "quarter": "Quarter", "brand": "Brand", "category_label": "Category",
        "total_revenue": "Revenue", "distinct_transactions": "Transactions",
        "item_rows": "Item Rows"
    }).drop(columns=["Brand"]).to_string(index=False))
    print()
print("⚠️  Q3 is partial (Jul 1 – Aug 6 only).")

In [0]:
# ═════════════════════════════════════════════════════════════════════════════
# CELL 6 — Quarterly Scene Penetration by Brand: Transactions & Revenue
# ═════════════════════════════════════════════════════════════════════════════

# --- Helper: add year/quarter columns ---
def _add_quarter(df):
    return (df
        .withColumn("year", (F.col("VisitDateId") / 10000).cast("int"))
        .withColumn("month", ((F.col("VisitDateId") / 100) % 100).cast("int"))
        .withColumn("quarter", F.concat(F.lit("Q"), F.ceil(F.col("month") / 3).cast("int"))))

def _agg_metrics_brand(df, prefix):
    """Aggregate transactions & revenue by year/quarter/brand, overall + by category."""
    base = _add_quarter(df)
    overall = (base.groupBy("year", "quarter", "brand")
        .agg(
            F.countDistinct("Transaction_Number").alias(f"{prefix}_txns"),
            F.sum(F.col("Net_Revenue").cast("double")).alias(f"{prefix}_rev"),
        ))
    by_cat = (base.groupBy("year", "quarter", "brand", "category_label")
        .agg(
            F.countDistinct("Transaction_Number").alias("txns"),
            F.sum(F.col("Net_Revenue").cast("double")).alias("rev"),
        )
        .groupBy("year", "quarter", "brand")
        .pivot("category_label", ["Food", "Games"])
        .agg(F.first("txns").alias("txns"), F.first("rev").alias("rev")))
    by_cat = (by_cat
        .withColumnRenamed("Food_txns", f"{prefix}_food_txns")
        .withColumnRenamed("Food_rev", f"{prefix}_food_rev")
        .withColumnRenamed("Games_txns", f"{prefix}_games_txns")
        .withColumnRenamed("Games_rev", f"{prefix}_games_rev"))
    return overall.join(by_cat, ["year", "quarter", "brand"], "inner")

# Aggregate for all transactions and Scene-only, by brand
all_metrics   = _agg_metrics_brand(pdm_tx, "total")
scene_metrics = _agg_metrics_brand(pdm_tx_scene, "scene")

# Join and compute ratios
result = (
    all_metrics.join(scene_metrics, ["year", "quarter", "brand"], "left")
    .withColumn("scene_txn_ratio", F.round(F.col("scene_txns") / F.col("total_txns") * 100, 1))
    .withColumn("scene_food_txn_ratio", F.round(F.col("scene_food_txns") / F.col("total_food_txns") * 100, 1))
    .withColumn("scene_games_txn_ratio", F.round(F.col("scene_games_txns") / F.col("total_games_txns") * 100, 1))
    .withColumn("scene_rev_ratio", F.round(F.col("scene_rev") / F.col("total_rev") * 100, 1))
    .withColumn("scene_food_rev_ratio", F.round(F.col("scene_food_rev") / F.col("total_food_rev") * 100, 1))
    .withColumn("scene_games_rev_ratio", F.round(F.col("scene_games_rev") / F.col("total_games_rev") * 100, 1))
    .orderBy("year", "quarter", "brand")
)

df_result = result.toPandas()

# --- Display per brand ---
for brand_name in ["Rec Room", "Playdium"]:
    df_b = df_result[df_result["brand"] == brand_name].copy()
    print("═" * 105)
    print(f"SCENE PENETRATION — TRANSACTIONS — {brand_name.upper()} (2025 & 2026)")
    print("═" * 105)
    txn_cols = df_b[["year", "quarter",
        "total_txns", "scene_txns", "scene_txn_ratio",
        "total_food_txns", "scene_food_txns", "scene_food_txn_ratio",
        "total_games_txns", "scene_games_txns", "scene_games_txn_ratio"]].copy()
    for c in ["total_txns","scene_txns","total_food_txns","scene_food_txns","total_games_txns","scene_games_txns"]:
        txn_cols[c] = txn_cols[c].map(lambda v: f"{int(v):,}" if pd.notna(v) else "—")
    for c in ["scene_txn_ratio","scene_food_txn_ratio","scene_games_txn_ratio"]:
        txn_cols[c] = txn_cols[c].map(lambda v: f"{v:.1f}%" if pd.notna(v) else "—")
    txn_cols.columns = ["Year","Qtr",
        "Total Txns","Scene Txns","Scene %",
        "Total F&B Txns","Scene F&B Txns","Scene F&B %",
        "Total Gaming Txns","Scene Gaming Txns","Scene Gaming %"]
    print(txn_cols.to_string(index=False))

    print("\n" + "═" * 105)
    print(f"SCENE PENETRATION — REVENUE — {brand_name.upper()} (2025 & 2026)")
    print("═" * 105)
    rev_cols = df_b[["year", "quarter",
        "total_rev", "scene_rev", "scene_rev_ratio",
        "total_food_rev", "scene_food_rev", "scene_food_rev_ratio",
        "total_games_rev", "scene_games_rev", "scene_games_rev_ratio"]].copy()
    for c in ["total_rev","scene_rev","total_food_rev","scene_food_rev","total_games_rev","scene_games_rev"]:
        rev_cols[c] = rev_cols[c].map(lambda v: f"${v:,.0f}" if pd.notna(v) else "—")
    for c in ["scene_rev_ratio","scene_food_rev_ratio","scene_games_rev_ratio"]:
        rev_cols[c] = rev_cols[c].map(lambda v: f"{v:.1f}%" if pd.notna(v) else "—")
    rev_cols.columns = ["Year","Qtr",
        "Total Rev","Scene Rev","Scene %",
        "Total F&B Rev","Scene F&B Rev","Scene F&B %",
        "Total Gaming Rev","Scene Gaming Rev","Scene Gaming %"]
    print(rev_cols.to_string(index=False))
    print()
print("⚠️  Q3 2026 is partial (Jul 1 – Aug 6 only).")

In [0]:
# ═════════════════════════════════════════════════════════════════════════════
# CELL 7 — Executive Charts: Scene Penetration by Brand (Rec Room vs Playdium)
# ═════════════════════════════════════════════════════════════════════════════
import matplotlib.pyplot as plt
import numpy as np

df_all = result.orderBy("year", "quarter", "brand").toPandas()
df_all["period"] = df_all.apply(
    lambda r: f"{int(r['year'])} {r['quarter']}\n⚠️ partial" if r["quarter"] == "Q3" and r["year"] == 2026 else f"{int(r['year'])} {r['quarter']}", axis=1)

# Colors
TXN_CLR = "#2C3E50"; REV_CLR = "#E74C3C"
FOOD_CLR = "#FF6B35"; GAME_CLR = "#1ABC9C"

plt.rcParams.update({"font.family": "sans-serif", "font.size": 10})

for brand_name in ["Rec Room", "Playdium"]:
    df = df_all[df_all["brand"] == brand_name].reset_index(drop=True)
    x = np.arange(len(df)); w = 0.35

    # ─── CHART 1: Overall Scene Penetration % ───
    fig, ax = plt.subplots(figsize=(14, 5.5))
    bars_txn = ax.bar(x - w/2, df["scene_txn_ratio"], w, color=TXN_CLR, label="Transaction %", edgecolor="white", zorder=3)
    bars_rev = ax.bar(x + w/2, df["scene_rev_ratio"], w, color=REV_CLR, label="Revenue %", edgecolor="white", zorder=3)
    for bar in bars_txn:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
                f"{bar.get_height():.0f}%", ha="center", fontsize=9, fontweight="bold", color=TXN_CLR)
    for bar in bars_rev:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
                f"{bar.get_height():.0f}%", ha="center", fontsize=9, fontweight="bold", color=REV_CLR)
    ax.set_xticks(x); ax.set_xticklabels(df["period"], fontsize=11)
    ax.set_ylabel("Scene Penetration (%)", fontsize=11)
    ax.set_title(f"{brand_name}: Overall Scene+ Penetration — Transactions vs Revenue",
                 fontsize=13, fontweight="bold", pad=12)
    ax.legend(fontsize=11, loc="upper left")
    ax.grid(axis="y", alpha=0.3, zorder=0); ax.set_axisbelow(True)
    ax.set_ylim(0, max(df["scene_txn_ratio"].max(), df["scene_rev_ratio"].max()) + 5)
    plt.tight_layout(); plt.show()

    # ─── CHART 2: F&B vs Gaming Penetration ───
    fig, axes = plt.subplots(1, 2, figsize=(16, 5.5))
    ax = axes[0]
    bars_txn = ax.bar(x - w/2, df["scene_food_txn_ratio"], w, color=FOOD_CLR, alpha=0.5, label="Txn %", edgecolor="white", zorder=3)
    bars_rev = ax.bar(x + w/2, df["scene_food_rev_ratio"], w, color=FOOD_CLR, label="Rev %", edgecolor="white", zorder=3)
    for bar in bars_txn:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.2,
                f"{bar.get_height():.0f}%", ha="center", fontsize=8, color="#333")
    for bar in bars_rev:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.2,
                f"{bar.get_height():.0f}%", ha="center", fontsize=8, fontweight="bold", color=FOOD_CLR)
    ax.set_xticks(x); ax.set_xticklabels(df["period"], fontsize=10)
    ax.set_title(f"{brand_name} — F&B Scene %\nTxn (light) vs Revenue (solid)", fontsize=12, fontweight="bold")
    ax.set_ylabel("Penetration (%)"); ax.legend(fontsize=10)
    ax.grid(axis="y", alpha=0.3, zorder=0); ax.set_axisbelow(True)
    ax.set_ylim(0, max(df["scene_food_rev_ratio"].max(), df["scene_food_txn_ratio"].max()) + 5)
    ax = axes[1]
    bars_txn = ax.bar(x - w/2, df["scene_games_txn_ratio"], w, color=GAME_CLR, alpha=0.5, label="Txn %", edgecolor="white", zorder=3)
    bars_rev = ax.bar(x + w/2, df["scene_games_rev_ratio"], w, color=GAME_CLR, label="Rev %", edgecolor="white", zorder=3)
    for bar in bars_txn:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.2,
                f"{bar.get_height():.0f}%", ha="center", fontsize=8, color="#333")
    for bar in bars_rev:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.2,
                f"{bar.get_height():.0f}%", ha="center", fontsize=8, fontweight="bold", color=GAME_CLR)
    ax.set_xticks(x); ax.set_xticklabels(df["period"], fontsize=10)
    ax.set_title(f"{brand_name} — Gaming Scene %\nTxn (light) vs Revenue (solid)", fontsize=12, fontweight="bold")
    ax.set_ylabel("Penetration (%)"); ax.legend(fontsize=10)
    ax.grid(axis="y", alpha=0.3, zorder=0); ax.set_axisbelow(True)
    ax.set_ylim(0, max(df["scene_games_txn_ratio"].max(), df["scene_games_rev_ratio"].max()) + 5)
    plt.suptitle(f"{brand_name}: F&B vs Gaming Scene Penetration — Quarterly (2025–2026)",
                 fontsize=13, fontweight="bold", y=1.02)
    plt.tight_layout(); plt.show()

    # ─── CHART 3: Revenue composition stacked ───
    fig, axes = plt.subplots(1, 2, figsize=(16, 5))
    ax = axes[0]
    non_scene_food = (df["total_food_rev"] - df["scene_food_rev"]) / 1e6
    scene_food = df["scene_food_rev"] / 1e6
    ax.bar(x, non_scene_food, 0.6, color="#BDC3C7", label="Non-Scene", edgecolor="white", zorder=3)
    ax.bar(x, scene_food, 0.6, bottom=non_scene_food, color=FOOD_CLR, label="Scene", edgecolor="white", zorder=3)
    for i, (ns, s) in enumerate(zip(non_scene_food, scene_food)):
        ax.text(i, ns + s + 0.05, f"${ns+s:.1f}M", ha="center", fontsize=8, fontweight="bold")
    ax.set_xticks(x); ax.set_xticklabels(df["period"], fontsize=10)
    ax.set_title(f"{brand_name} — F&B Revenue ($M)\nScene vs Non-Scene", fontsize=12, fontweight="bold")
    ax.set_ylabel("Revenue ($M)")
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"${v:.1f}M"))
    ax.legend(fontsize=9); ax.grid(axis="y", alpha=0.3, zorder=0); ax.set_axisbelow(True)
    ax = axes[1]
    non_scene_games = (df["total_games_rev"] - df["scene_games_rev"]) / 1e6
    scene_games = df["scene_games_rev"] / 1e6
    ax.bar(x, non_scene_games, 0.6, color="#BDC3C7", label="Non-Scene", edgecolor="white", zorder=3)
    ax.bar(x, scene_games, 0.6, bottom=non_scene_games, color=GAME_CLR, label="Scene", edgecolor="white", zorder=3)
    for i, (ns, s) in enumerate(zip(non_scene_games, scene_games)):
        ax.text(i, ns + s + 0.05, f"${ns+s:.1f}M", ha="center", fontsize=8, fontweight="bold")
    ax.set_xticks(x); ax.set_xticklabels(df["period"], fontsize=10)
    ax.set_title(f"{brand_name} — Gaming Revenue ($M)\nScene vs Non-Scene", fontsize=12, fontweight="bold")
    ax.set_ylabel("Revenue ($M)")
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"${v:.1f}M"))
    ax.legend(fontsize=9); ax.grid(axis="y", alpha=0.3, zorder=0); ax.set_axisbelow(True)
    plt.suptitle(f"{brand_name}: Revenue Composition — Quarterly (2025–2026)",
                 fontsize=13, fontweight="bold", y=1.02)
    plt.tight_layout(); plt.show()

print("\n⚠️  Q3 2026 is partial (Jul 1 – Aug 6). Treat as directional only.")

In [0]:
# ═════════════════════════════════════════════════════════════════════════════
# CELL 8 — Average Spend per Scene Member by Brand: F&B vs Gaming by Quarter
# ═════════════════════════════════════════════════════════════════════════════

avg_spend = (
    _add_quarter(pdm_tx_scene)
    .filter(F.col("CDE_ID").isNotNull())
    .groupBy("year", "quarter", "brand", "category_label")
    .agg(
        F.sum(F.col("Net_Revenue").cast("double")).alias("total_rev"),
        F.countDistinct("CDE_ID").alias("unique_members"),
    )
    .withColumn("avg_spend_per_member", F.round(F.col("total_rev") / F.col("unique_members"), 2))
    .orderBy("year", "quarter", "brand", "category_label")
)

df_avg = avg_spend.toPandas()

for brand_name in ["Rec Room", "Playdium"]:
    df_disp = df_avg[df_avg["brand"] == brand_name].copy()
    df_disp["total_rev"] = df_disp["total_rev"].map(lambda v: f"${v:,.0f}")
    df_disp["unique_members"] = df_disp["unique_members"].map(lambda v: f"{int(v):,}")
    df_disp["avg_spend_per_member"] = df_disp["avg_spend_per_member"].map(lambda v: f"${v:,.2f}")
    print("═" * 95)
    print(f"AVG SPEND PER SCENE MEMBER — {brand_name.upper()} — F&B vs Gaming (2025 & 2026)")
    print("═" * 95)
    print(df_disp.rename(columns={
        "year": "Year", "quarter": "Qtr", "brand": "Brand", "category_label": "Category",
        "total_rev": "Total Revenue", "unique_members": "Unique Members",
        "avg_spend_per_member": "Avg Spend/Member"
    }).drop(columns=["Brand"]).to_string(index=False))
    print()
print("⚠️  Q3 2026 is partial (Jul 1 – Aug 6 only).")

In [0]:
# ═════════════════════════════════════════════════════════════════════════════
# CELL 9 — Chart: Avg Spend per Scene Member by Brand (F&B vs Gaming)
# ═════════════════════════════════════════════════════════════════════════════
import matplotlib.pyplot as plt
import numpy as np

FOOD_CLR = "#FF6B35"; GAME_CLR = "#1ABC9C"
FOOD_LINE = "#C0392B"; GAME_LINE = "#16A085"

for brand_name in ["Rec Room", "Playdium"]:
    df_b = df_avg[df_avg["brand"] == brand_name]
    food = df_b[df_b["category_label"] == "Food"].reset_index(drop=True)
    games = df_b[df_b["category_label"] == "Games"].reset_index(drop=True)
    periods = food.apply(
        lambda r: f"{int(r['year'])} {r['quarter']}\n⚠️ partial" if r["quarter"] == "Q3" and r["year"] == 2026 else f"{int(r['year'])} {r['quarter']}", axis=1)

    fig, ax = plt.subplots(figsize=(14, 5.5))
    x = np.arange(len(periods)); w = 0.35

    bars_food = ax.bar(x - w/2, food["avg_spend_per_member"], w, color=FOOD_CLR, label="F&B Avg Spend", edgecolor="white", zorder=3)
    bars_games = ax.bar(x + w/2, games["avg_spend_per_member"], w, color=GAME_CLR, label="Gaming Avg Spend", edgecolor="white", zorder=3)

    for bar in bars_food:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                f"${bar.get_height():.0f}", ha="center", fontsize=9, fontweight="bold", color=FOOD_CLR)
    for bar in bars_games:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                f"${bar.get_height():.0f}", ha="center", fontsize=9, fontweight="bold", color=GAME_CLR)

    ax.set_xticks(x); ax.set_xticklabels(periods, fontsize=11)
    ax.set_ylabel("Avg Spend per Member ($)", fontsize=11)
    ax.set_ylim(0, max(food["avg_spend_per_member"].max(), games["avg_spend_per_member"].max()) + 12)
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"${v:.0f}"))
    ax.grid(axis="y", alpha=0.3, zorder=0); ax.set_axisbelow(True)

    ax2 = ax.twinx()
    ax2.plot(x, food["unique_members"] / 1000, marker="o", lw=2, color=FOOD_LINE, linestyle="--", label="F&B Members (K)", zorder=5)
    ax2.plot(x, games["unique_members"] / 1000, marker="D", lw=2, color=GAME_LINE, linestyle="--", label="Gaming Members (K)", zorder=5)
    ax2.set_ylabel("Unique Members (thousands)", fontsize=11)
    ax2.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.0f}K"))

    lines1, labels1 = ax.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax.legend(lines1 + lines2, labels1 + labels2, fontsize=9, loc="upper left", ncol=2)

    ax.set_title(f"{brand_name}: Avg Quarterly Spend per Scene Member + Member Volume\nF&B vs Gaming (2025–2026)",
                 fontsize=13, fontweight="bold", pad=12)

    plt.tight_layout(); plt.show()

In [0]:
# ═════════════════════════════════════════════════════════════════════════════
# Visit-level composition: classify each visit (CDE_ID, VisitDateId, LocationId)
# as F&B Only, Gaming Only, or Both — then show member shares per quarter.
# ═════════════════════════════════════════════════════════════════════════════
import matplotlib.pyplot as plt
import numpy as np

# Classify each visit by what was purchased
visit_cats = (
    pdm_tx_scene
    .filter(F.col("CDE_ID").isNotNull())
    .select("CDE_ID", "VisitDateId", "LocationId", "brand", "category_label")
    .distinct()
    .groupBy("CDE_ID", "VisitDateId", "LocationId", "brand")
    .agg(F.collect_set("category_label").alias("cats"))
    .withColumn("visit_type",
        F.when(F.array_contains(F.col("cats"), "Food") & F.array_contains(F.col("cats"), "Games"), F.lit("Both"))
         .when(F.array_contains(F.col("cats"), "Food"), F.lit("F&B Only"))
         .otherwise(F.lit("Gaming Only")))
    .withColumn("year", (F.col("VisitDateId") / 10000).cast("int"))
    .withColumn("month", ((F.col("VisitDateId") / 100) % 100).cast("int"))
    .withColumn("quarter", F.concat(F.lit("Q"), F.ceil(F.col("month") / 3).cast("int")))
)

# Aggregate: total visits + visit counts by type per quarter/brand
total_visits = (
    visit_cats.groupBy("year", "quarter", "brand")
    .agg(F.count("*").alias("total_visits"),
         F.countDistinct("CDE_ID").alias("total_members"))
)

visit_shares = (
    visit_cats.groupBy("year", "quarter", "brand", "visit_type")
    .agg(F.count("*").alias("visits"))
)

df_totals = total_visits.orderBy("year", "quarter", "brand").toPandas()
df_shares = visit_shares.orderBy("year", "quarter", "brand", "visit_type").toPandas()

# Pivot to get shares
df_pivot = df_shares.pivot_table(index=["year", "quarter", "brand"], columns="visit_type",
                                  values="visits", fill_value=0).reset_index()
df_pivot = df_pivot.merge(df_totals, on=["year", "quarter", "brand"])

# Compute percentages (visit-based — sums to 100%)
for vt in ["F&B Only", "Gaming Only", "Both"]:
    df_pivot[f"{vt} %"] = (df_pivot[vt] / df_pivot["total_visits"] * 100).round(1)

df_pivot["period"] = df_pivot.apply(
    lambda r: f"{int(r['year'])} {r['quarter']}\n⚠️ partial" if r["quarter"] == "Q3" and r["year"] == 2026
    else f"{int(r['year'])} {r['quarter']}", axis=1)


In [0]:

# ─── CHART ───
CLR_BOTH = "#8E44AD"; CLR_FOOD = "#FF6B35"; CLR_GAME = "#1ABC9C"

for brand_name in ["Rec Room", "Playdium"]:
    df = df_pivot[df_pivot["brand"] == brand_name].reset_index(drop=True)
    x = np.arange(len(df))

    fig, ax1 = plt.subplots(figsize=(14, 6))

    # Stacked bars for shares
    b1 = ax1.bar(x, df["Both %"], 0.55, color=CLR_BOTH, label="Both (F&B + Gaming)", edgecolor="white", zorder=3)
    b2 = ax1.bar(x, df["F&B Only %"], 0.55, bottom=df["Both %"], color=CLR_FOOD, label="F&B Only", edgecolor="white", zorder=3)
    b3 = ax1.bar(x, df["Gaming Only %"], 0.55, bottom=df["Both %"] + df["F&B Only %"],
                 color=CLR_GAME, label="Gaming Only", edgecolor="white", zorder=3)

    # Data labels on all segments
    for i, v in enumerate(df["Both %"]):
        ax1.text(i, v/2, f"{v:.0f}%", ha="center", va="center", fontsize=9, fontweight="bold", color="white")
    for i, (both, food) in enumerate(zip(df["Both %"], df["F&B Only %"])):
        ax1.text(i, both + food/2, f"{food:.0f}%", ha="center", va="center", fontsize=9, fontweight="bold", color="white")
    for i, (both, food, game) in enumerate(zip(df["Both %"], df["F&B Only %"], df["Gaming Only %"])):
        ax1.text(i, both + food + game/2, f"{game:.0f}%", ha="center", va="center", fontsize=9, fontweight="bold", color="white")

    ax1.set_xticks(x); ax1.set_xticklabels(df["period"], fontsize=10)
    ax1.set_ylabel("Share of Visits (%)", fontsize=11)
    ax1.set_ylim(0, 105)
    ax1.legend(loc="upper left", fontsize=10)
    ax1.grid(axis="y", alpha=0.3, zorder=0); ax1.set_axisbelow(True)

    # Secondary axis: total visit count
    ax2 = ax1.twinx()
    ax2.plot(x, df["total_visits"] / 1000, "ko-", markersize=6, linewidth=2, label="Total Visits (K)")
    for i, v in enumerate(df["total_visits"]):
        ax2.text(i, v/1000 + 0.5, f"{v/1000:.1f}K", ha="center", fontsize=8, color="#333")
    ax2.set_ylabel("Total Scene Visits (K)", fontsize=11)
    ax2.legend(loc="upper right", fontsize=10)

    ax1.set_title(f"{brand_name}: Scene Visit Composition — What Gets Purchased?"
                  ,
                  fontsize=13, fontweight="bold", pad=12)
    plt.tight_layout(); plt.show()

    # Print summary
    print(f"\n{brand_name} — Visit shares by type:")
    print(df[["period", "total_visits", "total_members", "Both %", "F&B Only %", "Gaming Only %"]].to_string(index=False))
    print()

In [0]:
# ═════════════════════════════════════════════════════════════════════════════
# Monthly Visit Composition: F&B Only / Gaming Only / Both
# Reuses visit_cats from Cell 10 (already has year, month columns)
# ═════════════════════════════════════════════════════════════════════════════
import matplotlib.pyplot as plt
import numpy as np

# Monthly aggregation
monthly_total = (
    visit_cats.groupBy("year", "month", "brand")
    .agg(F.count("*").alias("total_visits"),
         F.countDistinct("CDE_ID").alias("total_members"))
)
monthly_shares = (
    visit_cats.groupBy("year", "month", "brand", "visit_type")
    .agg(F.count("*").alias("visits"))
)

df_m_total = monthly_total.orderBy("year", "month", "brand").toPandas()
df_m_shares = monthly_shares.orderBy("year", "month", "brand", "visit_type").toPandas()

df_m = df_m_shares.pivot_table(index=["year", "month", "brand"], columns="visit_type",
                                values="visits", fill_value=0).reset_index()
df_m = df_m.merge(df_m_total, on=["year", "month", "brand"])

for vt in ["F&B Only", "Gaming Only", "Both"]:
    df_m[f"{vt} %"] = (df_m[vt] / df_m["total_visits"] * 100).round(1)

df_m["period"] = df_m.apply(
    lambda r: f"{int(r['year'])}-{int(r['month']):02d}", axis=1)


In [0]:

# Mark Aug 2026 as partial
df_m.loc[(df_m["year"] == 2026) & (df_m["month"] == 8), "period"] = "2026-08\n⚠️ partial"

# ─── CHART ───
CLR_BOTH = "#8E44AD"; CLR_FOOD = "#FF6B35"; CLR_GAME = "#1ABC9C"

for brand_name in ["Rec Room", "Playdium"]:
    df = df_m[df_m["brand"] == brand_name].reset_index(drop=True)
    x = np.arange(len(df))

    fig, ax1 = plt.subplots(figsize=(18, 6))

    # Stacked bars
    ax1.bar(x, df["Both %"], 0.7, color=CLR_BOTH, label="Both (F&B + Gaming)", edgecolor="white", zorder=3)
    ax1.bar(x, df["F&B Only %"], 0.7, bottom=df["Both %"], color=CLR_FOOD, label="F&B Only", edgecolor="white", zorder=3)
    ax1.bar(x, df["Gaming Only %"], 0.7, bottom=df["Both %"] + df["F&B Only %"],
            color=CLR_GAME, label="Gaming Only", edgecolor="white", zorder=3)

    # Labels on Both segment
    for i, v in enumerate(df["Both %"]):
        ax1.text(i, v/2, f"{v:.0f}%", ha="center", va="center", fontsize=7, fontweight="bold", color="white")
    # Labels on F&B Only
    for i, (both, food) in enumerate(zip(df["Both %"], df["F&B Only %"])):
        ax1.text(i, both + food/2, f"{food:.0f}%", ha="center", va="center", fontsize=7, fontweight="bold", color="white")
    # Labels on Gaming Only
    for i, (both, food, game) in enumerate(zip(df["Both %"], df["F&B Only %"], df["Gaming Only %"])):
        ax1.text(i, both + food + game/2, f"{game:.0f}%", ha="center", va="center", fontsize=7, fontweight="bold", color="white")

    ax1.set_xticks(x); ax1.set_xticklabels(df["period"], fontsize=8, rotation=45, ha="right")
    ax1.set_ylabel("Share of Visits (%)", fontsize=11)
    ax1.set_ylim(0, 105)
    ax1.legend(loc="upper left", fontsize=10)
    ax1.grid(axis="y", alpha=0.3, zorder=0); ax1.set_axisbelow(True)

    # Secondary axis: total visits
    ax2 = ax1.twinx()
    ax2.plot(x, df["total_visits"] / 1000, "ko-", markersize=5, linewidth=1.8, label="Total Visits (K)")
    for i, v in enumerate(df["total_visits"]):
        if i % 2 == 0:  # label every other point to avoid clutter
            ax2.text(i, v/1000 + 0.4, f"{v/1000:.1f}K", ha="center", fontsize=7, color="#333")
    ax2.set_ylabel("Total Scene Visits (K)", fontsize=11)
    ax2.legend(loc="upper right", fontsize=10)

    ax1.set_title(f"{brand_name}: Monthly Scene Visit Composition",
                  fontsize=13, fontweight="bold", pad=12)
    plt.tight_layout(); plt.show()

    print(f"\n{brand_name} — Monthly visit shares:")
    print(df[["period", "total_visits", "total_members", "Both %", "F&B Only %", "Gaming Only %"]].to_string(index=False))
    print()

## Key Insights: F&B & Gaming Scene Penetration by Brand (2025-2026)

---

### The Rec Room (TRR)

**1. Scene Revenue Share Growing Faster Than Transaction Share**
- Overall Scene transaction penetration stable at **24-26%**; revenue penetration climbing from 22.4% to **25.5%** (Q1 YoY).
- The gap = Scene members are spending more per visit than non-Scene guests.
- F&B Scene revenue penetration jumped **15.3% to 21.7%** (Q1 YoY) while transaction penetration grew 9.2% to 13.4%.

**2. F&B Cross-Purchase Adoption is the Growth Driver**
- Unique F&B Scene members grew **+23%** YoY in Q1 (19.2K to 23.6K).
- Gaming member counts are larger (49-65K) but relatively flat.
- More Scene gamers at TRR are adding food to their visits.

**3. High Per-Member F&B Spend (~$97-107)**
- TRR delivers a full dining experience: avg quarterly F&B spend per Scene member is **$96-107** (peaks Q4 at $107).
- Gaming spend per member is stable at ~$80-83.
- F&B penetration growth is driven by **more members transacting**, not bigger baskets.

**4. Seasonality**
- **Q1/Q3**: Penetration peaks (loyal members form larger share as casual traffic dips).
- **Q4**: Holiday influx dilutes the ratio (denominator effect, not a Scene drop-off).

---

### Playdium (PDM)

**1. Highest Scene Penetration of Both Brands (31-36%)**
- Scene transaction penetration consistently **~10pp higher** than TRR (34.5% vs 24.6% in Q1 2025).
- Revenue penetration also elevated at **30-35%**.
- Playdium's loyalty-engaged guest base is proportionally much larger.

**2. F&B Scene Penetration Strong on Transactions, Moderate on Revenue**
- F&B transaction penetration: **20-26%** (double TRR's rate).
- F&B revenue penetration: 15-25% (similar range to TRR despite much higher txn share).
- The gap is explained by spend levels (see #3).

**3. Low F&B Spend per Member (~$26-30) = Snack-Oriented Model**
- PDM F&B is grab-and-go / snack-focused: avg $26-30 per member vs TRR's $97-107.
- Gaming spend per member is $73-85 (only slightly below TRR).
- This positions PDM F&B as a **frequency play** (many members, small tickets) vs TRR's higher-value dining.

**4. Gaming Penetration Comparable to TRR**
- Scene gaming transaction penetration: **38-42%** (nearly identical to TRR's 37-41%).
- Revenue penetration: **30-37%** (slightly higher than TRR's 25-28%).

---

### Cross-Brand Comparison (Q1 2026)

| Metric | Rec Room | Playdium |
| --- | --- | --- |
| Scene Txn Penetration | 26.3% | 34.5% |
| Scene Rev Penetration | 25.5% | 33.0% |
| F&B Scene Rev % | 21.7% | 20.7% |
| Gaming Scene Rev % | 27.9% | 35.2% |
| Avg F&B Spend/Member | $99.76 | $28.30 |
| Avg Gaming Spend/Member | $83.42 | $76.39 |
| Unique F&B Members | 23.6K | 5.4K |
| Unique Gaming Members | 60.7K | 18.7K |

---

**Bottom Line:**
- **TRR opportunity**: Existing Scene base is increasingly converting into F&B purchasers (cross-purchase growth), driving revenue penetration up without bigger baskets. F&B marketing should target cross-sell from gaming-only Scene members.
- **PDM opportunity**: Already high Scene penetration, but low F&B ticket size. Growth lever is **basket upsell** (combos, premiums) rather than new member acquisition. Gaming revenue capture is already best-in-class.

*Q3 2026 is partial (Jul 1 - Aug 6). Treat Q3 figures as directional only.*